# Perturbative Expansion About A Fixed Point, No DW

This notebook validates the handwritten rank-1 perturbative expansion in the no-domain-wall setting.

For a rank-1 projector `P = |chi><chi|`, write `Q = I - P` and let `s = +1` denote the occupied/particle target and `s = -1` denote the unoccupied/hole target. The exact top-layer measurement map used here is

`M_s(G) = s P + Q G Q - s Q G P G Q / (1 + s Tr(G P))`.

If `Gstar` is a fixed point of this rank-1 update, then it must have the target covariance in the measured mode: `Gstar P = P Gstar = s P`. Expanding `G = Gstar + eps dG` gives

`M_s(Gstar + eps dG) = Gstar + eps Q dG Q + O(eps^2)`.

So the linearized perturbation update is `dG -> Q dG Q`. This is the statement tested below for both target outcomes, and then against the repository implementation with `DW=False`.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from fgtn.classA_U1FGTN import classA_U1FGTN

np.set_printoptions(precision=3, suppress=True)
print(f"repo root: {REPO_ROOT}")

## Helpers

In [ ]:
def hermitize(A):
    return 0.5 * (A + A.conj().T)


def fro_norm(A):
    return float(np.linalg.norm(A, ord="fro"))


def random_complex_matrix(n, m, rng):
    return rng.normal(size=(n, m)) + 1j * rng.normal(size=(n, m))


def random_hermitian(n, rng):
    A = random_complex_matrix(n, n, rng)
    H = hermitize(A)
    return H / fro_norm(H)


def rank1_projector_from_vector(v):
    v = np.asarray(v, dtype=np.complex128).reshape(-1)
    v = v / np.linalg.norm(v)
    return np.outer(v, v.conj())


def random_rank1_projector(n, rng):
    v = random_complex_matrix(n, 1, rng).reshape(-1)
    return rank1_projector_from_vector(v)


def orthonormal_complement_from_projector(P, rng):
    evals, evecs = np.linalg.eigh(hermitize(P))
    u = evecs[:, np.argmax(evals)]
    Z = random_complex_matrix(P.shape[0], P.shape[0] - 1, rng)
    basis, _ = np.linalg.qr(np.column_stack([u, Z]))
    return basis[:, 1:]


def make_rank1_fixed_covariance(P, s, rng):
    """Return Gstar with Gstar P = P Gstar = s P and Gstar^2 = I."""
    B = orthonormal_complement_from_projector(P, rng)
    signs = rng.choice([-1.0, 1.0], size=B.shape[1])
    Gq = B @ np.diag(signs) @ B.conj().T
    return hermitize(s * P + Gq)


def measurement_map_closed_form(G, P, s):
    """Closed-form rank-1 measurement map. s=+1 occupied, s=-1 unoccupied."""
    I = np.eye(G.shape[0], dtype=np.complex128)
    Q = I - P
    denom = 1.0 + s * np.trace(G @ P)
    out = s * P + Q @ G @ Q - s * (Q @ G @ P @ G @ Q) / denom
    return hermitize(out)


def measurement_map_solve_form(G, P, s):
    """Same map written in the Schur-complement form used in classA_U1FGTN."""
    I = np.eye(G.shape[0], dtype=np.complex128)
    Q = I - P
    H11 = -s * P
    H22 = s * P
    A = G @ H11 - I
    B = G @ Q
    out = H22 - Q @ np.linalg.solve(A, B)
    return hermitize(out)


def print_error_table(rows):
    header = "eps          ||resid||      ||resid||/eps^2   ||fd-lin||"
    print(header)
    print("-" * len(header))
    for eps, resid, resid_over_eps2, fd_err in rows:
        print(f"{eps:1.0e}    {resid:12.4e}    {resid_over_eps2:14.4e}    {fd_err:12.4e}")

First, verify that the Sherman-Morrison closed form and the direct solve form agree away from the fixed point.

In [ ]:
rng = np.random.default_rng(1234)
n = 12
P = random_rank1_projector(n, rng)
G = random_hermitian(n, rng)

for s in (+1, -1):
    closed = measurement_map_closed_form(G, P, s)
    solved = measurement_map_solve_form(G, P, s)
    err = fro_norm(closed - solved)
    print(f"s={s:+d}: closed form vs solve form error = {err:.3e}")
    assert err < 1e-10

## Linearization For A Single Rank-1 Update

At a fixed point satisfying `Gstar P = P Gstar = s P`, the first-order prediction is `dG_next = Q dG Q`. The residual below is

`||M_s(Gstar + eps dG) - Gstar - eps Q dG Q||`.

For a correct first-order expansion this residual should scale as `eps^2`, while the finite-difference error should scale as `eps`.

In [ ]:
def single_update_linearization_table(s, seed=0):
    rng = np.random.default_rng(seed)
    n = 16
    P = random_rank1_projector(n, rng)
    Q = np.eye(n, dtype=np.complex128) - P
    Gstar = make_rank1_fixed_covariance(P, s, rng)
    dG = random_hermitian(n, rng)
    lin = Q @ dG @ Q

    fixed_err = fro_norm(measurement_map_closed_form(Gstar, P, s) - Gstar)
    comm_err = fro_norm(Gstar @ P - P @ Gstar)
    target_err = fro_norm(Gstar @ P - s * P)
    print(f"s={s:+d}: fixed_err={fixed_err:.3e}, comm_err={comm_err:.3e}, target_err={target_err:.3e}")

    rows = []
    for eps in [1e-1, 3e-2, 1e-2, 3e-3, 1e-3, 3e-4, 1e-4]:
        exact = measurement_map_closed_form(Gstar + eps * dG, P, s)
        resid = fro_norm(exact - Gstar - eps * lin)
        fd_err = fro_norm((exact - Gstar) / eps - lin)
        rows.append((eps, resid, resid / eps**2, fd_err))
    return rows


rows_occ = single_update_linearization_table(+1, seed=5)
print_error_table(rows_occ)
print()
rows_unocc = single_update_linearization_table(-1, seed=6)
print_error_table(rows_unocc)

assert rows_occ[-1][3] < 1e-4
assert rows_unocc[-1][3] < 1e-4

Plotting the residual confirms the expected second-order scaling.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for label, rows in [("occupied s=+1", rows_occ), ("unoccupied s=-1", rows_unocc)]:
    eps = np.array([row[0] for row in rows])
    resid = np.array([row[1] for row in rows])
    ax.loglog(eps, resid, "o-", label=label)
ax.loglog(eps, resid[-1] * (eps / eps[-1]) ** 2, "k--", lw=1, label="slope 2")
ax.set_xlabel("eps")
ax.set_ylabel("linearization residual")
ax.legend()
ax.grid(True, which="both", alpha=0.25)
fig.tight_layout()

## What Fails Away From A Fixed Point

The simplification to `Q dG Q` uses the fixed-point identities `Gstar P = P Gstar = s P`. If those identities are not true, the derivative contains additional terms from `Q G P G Q` and from the denominator.

In [ ]:
rng = np.random.default_rng(77)
n = 12
s = +1
P = random_rank1_projector(n, rng)
Q = np.eye(n, dtype=np.complex128) - P
Gref = random_hermitian(n, rng)
dG = random_hermitian(n, rng)
eps = 1e-6

fd_deriv = (measurement_map_closed_form(Gref + eps * dG, P, s) - measurement_map_closed_form(Gref, P, s)) / eps
naive = Q @ dG @ Q
print(f"away from fixed point, derivative mismatch = {fro_norm(fd_deriv - naive):.3e}")
assert fro_norm(fd_deriv - naive) > 1e-3

## Repository No-DW Single-Wannier-Projector Check

Now use the actual project Wannier projector with `DW=False`. The projector is built as `P = |W><W|` from one stored overcomplete Wannier spinor, for example `WF_Ap[:, Rx, Ry]` or `WF_Am[:, Rx, Ry]`.

This section validates one projector at a time, matching the handwritten derivation directly.

In [ ]:
model = classA_U1FGTN(Nx=4, Ny=4, DW=False, nshell=None, alpha_1=1, alpha_2=1)
model.construct_OW_projectors(nshell=None, DW=False)
Nlayer = model.Ntot // 2

def repo_single_wannier_table(WF, Rx, Ry, s, seed):
    rng = np.random.default_rng(seed)
    P = model._proj_from_WF(WF, Rx, Ry)
    P = hermitize(P)
    Q = np.eye(Nlayer, dtype=np.complex128) - P
    Gstar = make_rank1_fixed_covariance(P, s, rng)
    dG = random_hermitian(Nlayer, rng)
    lin = Q @ dG @ Q

    def repo_update(G):
        return model.measure_only_top_layer(G, P, particle=(s == +1))

    fixed_err = fro_norm(repo_update(Gstar) - Gstar)
    target_err = fro_norm(Gstar @ P - s * P)
    print(f"single Wannier projector, s={s:+d}: fixed_err={fixed_err:.3e}, target_err={target_err:.3e}")
    assert fixed_err < 1e-10

    rows = []
    for eps in [1e-1, 3e-2, 1e-2, 3e-3, 1e-3, 3e-4, 1e-4]:
        exact = repo_update(Gstar + eps * dG)
        resid = fro_norm(exact - Gstar - eps * lin)
        fd_err = fro_norm((exact - Gstar) / eps - lin)
        rows.append((eps, resid, resid / eps**2, fd_err))
    print_error_table(rows)
    assert rows[-1][3] < 1e-4
    return rows


# Ap is used as an unoccupied target in the project post-selection convention.
rows_repo_wannier_unocc = repo_single_wannier_table(model.WF_Ap, Rx=0, Ry=0, s=-1, seed=2026)
print()
# Am is used as an occupied target in the project post-selection convention.
rows_repo_wannier_occ = repo_single_wannier_table(model.WF_Am, Rx=0, Ry=0, s=+1, seed=2027)

## Conclusion

The perturbative result is correct for one rank-1 Wannier projection provided the expansion point is a fixed point of that same projector/outcome: `Gstar P = P Gstar = s P`. In that case the linearized perturbation is simply projected onto the unmeasured subspace, `dG_next = (I-P) dG (I-P)`. The no-DW project-code check above uses a single stored Wannier spinor and validates this directly for both unoccupied and occupied target outcomes.